In [ ]:
# Section 1: Imports and Setup
import os
import re
import math
import argparse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import MultiComparison
import spacy
from spacy.language import Language
from spacy.tokens import Doc
from spacy.matcher import PhraseMatcher, Matcher
import torch
torch.device('mps')
from collections import Counter
from typing import Dict, List, Set, Tuple
from tqdm import tqdm

# Regex pattern: remove all non-word characters except whitespace.
pattern = r'[^\w\s]'

# Configure pandas display option.
pd.set_option('display.max_colwidth', None)


spacy.prefer_gpu()
nlp = spacy.load('en_core_web_trf')

In [ ]:
# Section 2: Marker Definitions and Context Patterns
# Using Hyland's categorization scheme

INTERACTIVE_MARKERS = {
    "transitions": [
        "moreover", "furthermore", "in addition", "additionally", "besides", "similarly", 
        "likewise", "equally", "also", 
        "therefore", "thus", "consequently", "hence", "as a result", "because", "since", 
        "due to", "owing to", "so", 
        "however", "nevertheless", "nonetheless", "but", "yet", "though", "although", "even though", 
        "despite", "in spite of", "in contrast", "on the other hand", "conversely", 
        "meanwhile", "simultaneously", "subsequently", "previously", "after", "before", 
        "then", "later", "formerly", "eventually"
    ],
    "frame_markers": [
        "first", "firstly", "second", "secondly", "third", "thirdly", "fourth", "finally", 
        "lastly", "to begin with", "to start with", "next", "then", "subsequently",
        "in conclusion", "to conclude", "to summarize", "in summary", "in brief", "all in all", 
        "on the whole", "so far", "at this point", "overall",
        "aim", "purpose", "goal", "objective", "focus", "seek to", "intend to",
        "with regard to", "concerning", "regarding", "turning to", "moving on to", "back to"
    ],
    "endophoric_markers": [
        "in chapter", "in section", "in part", "in figure", "in table", "figure", "table", 
        "above", "below", "earlier", "previously", "as noted above", "as mentioned earlier", 
        "see", "refer to", "page", "the following", "as follows", "aforementioned"
    ],
    "evidentials": [
        "according to", "cited", "quoted", "states that", "argues that", "notes that", 
        "suggests that", "reports that", "found that", "observed that", "concluded that", 
        "in the literature", "previous research", "research shows", "studies indicate"
    ],
    "code_glosses": [
        "in other words", "that is", "i.e.", "that is to say", "this means", "in simple terms",
        "put simply", "to put it simply", "namely", "for example", "for instance", "such as", 
        "e.g.", "specifically", "particularly", "in fact", "indeed", "actually", "called", 
        "defined as", "referred to as", "including", "included", "especially", "notably"
    ]
}

INTERACTIONAL_MARKERS = {
    "hedges": [
        "may", "might", "could", "would", "perhaps", "possibly", "probably", "maybe", "likely", 
        "seemingly", "apparently", "approximately", "about", "roughly", "suggest", "assume", 
        "believe", "think", "appear", "seem", "indicate", "suspect", "suppose", "estimate", 
        "in my opinion", "from my perspective", "to my knowledge", "generally", "usually", 
        "sometimes", "often", "in most cases", "to some extent", "sort of", "kind of"
    ],
    "boosters": [
        "clearly", "obviously", "certainly", "definitely", "undoubtedly", "undeniably", 
        "demonstrate", "prove", "show", "establish", "confirm", "find", "reveal", "must", 
        "will", "beyond doubt", "without doubt", "in fact", "indeed", "actually", "always", 
        "never", "absolutely", "completely", "entirely", "truly", "really", 
        "it is clear that", "we found that", "we proved that"
    ],
    "attitude_markers": [
        "unfortunately", "fortunately", "surprisingly", "remarkably", "interestingly", 
        "hopefully", "importantly", "significantly", "correctly", "appropriately", "agree", 
        "prefer", "disagree", "dramatic", "unexpected", "desirable", "disappointing", "alarming",
        "it is surprising that", "it is important that", "it is significant that"
    ],
    "engagement_markers": [
        "you", "your", "yours", "yourself", "consider", "note", "imagine", "think about", 
        "let us", "let's", "see", "must", "should", "need to", "have to", "ought to", 
        "what about", "how about", "by the way", "the reader", "readers"
    ],
    "self_mentions": [
        "i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves", 
        "the author", "the authors", "the researcher", "the researchers", "this author"
    ]
}

# Context rules to filter false positives.
CONTEXT_PATTERNS = {
    "code_glosses": [
        # Reformulations
        [{"LOWER": "in"}, {"LOWER": "other"}, {"LOWER": "words"}],
        [{"LOWER": "that"}, {"LOWER": "is"}, {"LOWER": {"IN": ["to", ""]}}, {"LOWER": "say", "OP": "?"}],
        [{"LOWER": {"IN": ["i.e.", "ie"]}}, {"IS_PUNCT": True, "OP": "?"}],
        [{"LOWER": "this"}, {"LOWER": "means"}],
        [{"LOWER": "put"}, {"LOWER": "simply"}],
        [{"LOWER": "namely"}],
        
        # Examples
        [{"LOWER": "for"}, {"LOWER": "example"}],
        [{"LOWER": "for"}, {"LOWER": "instance"}],
        [{"LOWER": "such"}, {"LOWER": "as"}],
        [{"LOWER": {"IN": ["e.g.", "eg"]}}, {"IS_PUNCT": True, "OP": "?"}],
        [{"LOWER": "including"}],
        
        # Clarifications
        [{"LOWER": "specifically"}],
        [{"LOWER": "particularly"}],
        [{"LOWER": "in"}, {"LOWER": "fact"}],
        [{"LOWER": "indeed"}],
    ],
    "boosters": [
        # Certainty markers
        [{"LOWER": {"IN": ["clearly", "obviously", "certainly", "definitely", "undoubtedly"]}}, 
         {"IS_PUNCT": True, "OP": "?"}],
        [{"LOWER": "beyond"}, {"LOWER": "doubt"}],
        [{"LOWER": "without"}, {"LOWER": "doubt"}],
        
        # Emphatic verbs
        [{"LOWER": {"IN": ["demonstrate", "demonstrates", "demonstrated", "demonstrates"]}}, 
         {"LOWER": "that", "OP": "?"}],
        [{"LOWER": {"IN": ["prove", "proves", "proved", "proven"]}}, 
         {"LOWER": "that", "OP": "?"}],
        [{"LOWER": {"IN": ["show", "shows", "showed", "shown"]}}, 
         {"LOWER": "that", "OP": "?"}],
        [{"LOWER": {"IN": ["establish", "establishes", "established"]}}, 
         {"LOWER": "that", "OP": "?"}],
        
        # Intensifiers in context
        [{"LOWER": "it"}, {"LOWER": "is"}, {"LOWER": "clear"}, {"LOWER": "that"}],
        [{"LOWER": "we"}, {"LOWER": {"IN": ["found", "discovered", "proved"]}}, {"LOWER": "that"}],
        [{"LOWER": "must"}, {"TAG": "VB"}],
    ],
    "attitude_markers": [
        # Evaluative adverbs
        [{"LOWER": {"IN": ["unfortunately", "fortunately", "surprisingly", "remarkably", 
                      "interestingly", "importantly", "significantly"]}}, 
         {"IS_PUNCT": True, "OP": "?"}],
        
        # Attitudinal expressions
        [{"LOWER": "it"}, {"LOWER": "is"}, 
         {"LOWER": {"IN": ["surprising", "important", "interesting", "remarkable", "significant"]}}, 
         {"LOWER": "that"}],
        
        # Evaluative adjectives in specific contexts
        [{"POS": "PRON"}, {"LEMMA": "be"}, 
         {"LOWER": {"IN": ["glad", "happy", "pleased", "disappointed", "concerned", "alarmed"]}}],
        
        # Agreement/disagreement
        [{"LOWER": {"IN": ["i", "we"]}}, 
         {"LOWER": {"IN": ["agree", "disagree", "prefer"]}}],
    ],
    "endophoric_markers": [
        # Figure/table references
        [{"LOWER": {"IN": ["figure", "fig", "fig."]}}, {"IS_DIGIT": True}],
        [{"LOWER": {"IN": ["table", "tbl", "tbl."]}}, {"IS_DIGIT": True}],
        [{"LOWER": {"IN": ["appendix", "app", "app."]}}, {"IS_DIGIT": True, "OP": "?"}],
        
        # Text section references
        [{"LOWER": "in"}, {"LOWER": {"IN": ["chapter", "section", "part"]}}, {"IS_DIGIT": True, "OP": "?"}],
        [{"LOWER": "the"}, {"LOWER": {"IN": ["previous", "next", "following", "preceding"]}}, 
         {"LOWER": {"IN": ["chapter", "section", "paragraph", "part"]}}],
        
        # Directional references
        [{"LOWER": {"IN": ["above", "below", "earlier", "previously", "later"]}}, 
         {"IS_PUNCT": True, "OP": "?"}],
        [{"LOWER": "as"}, {"LOWER": {"IN": ["noted", "mentioned", "discussed", "shown"]}}, 
         {"LOWER": {"IN": ["above", "below", "earlier"]}}],
    ],
    "engagement_markers": [
        # Direct address
        [{"LOWER": {"IN": ["you", "your", "yourself"]}}, {"POS": {"NOT_IN": ["NOUN"]}}],
        
        # Imperatives
        [{"IS_SENT_START": True}, {"LOWER": {"IN": ["consider", "note", "imagine", "see", "suppose"]}}, 
         {"IS_PUNCT": True, "OP": "?"}],
        
        # Questions and inclusive expressions
        [{"LOWER": {"IN": ["what", "how"]}}, {"LOWER": "about"}],
        [{"LOWER": "let"}, {"LOWER": {"IN": ["us", "'s", "s"]}}],
        
        # Obligation in context
        [{"LOWER": {"IN": ["must", "should", "need", "ought"]}}, {"LOWER": "to", "OP": "?"}, {"TAG": "VB"}],
        
        # Reader references
        [{"LOWER": "the"}, {"LOWER": {"IN": ["reader", "readers"]}}],
    ],
    "hedges": [
        [{"LOWER": {"IN": ["may", "might", "could", "would"]}},
         {"TAG": {"NOT_IN": ["NN", "NNP"]}, "OP": "+"},
         {"TAG": "VB"}],
        [{"LOWER": {"IN": ["i", "we"]}},
         {"LOWER": {"IN": ["think", "believe", "assume", "suppose"]}}],
        [{"LOWER": "it"},
         {"LOWER": {"IN": ["appears", "seems", "looks"]}}],
        [{"LOWER": "likely"}, {"LOWER": "to"}],
    ],
    "frame_markers": [
        [{"LOWER": {"IN": ["first", "second", "third", "finally", "lastly"]}},
         {"IS_PUNCT": True, "LOWER": ","}],
        [{"LOWER": {"IN": ["i", "we", "this", "the"]}},
         {"LOWER": {"IN": ["aim", "intend", "focus", "purpose"]}}],
        [{"LOWER": {"IN": ["turning", "moving"]}},
         {"LOWER": "to"}],
    ],
    "transitions": [
        [{"IS_SENT_START": True},
         {"LOWER": {"IN": ["however", "nevertheless", "thus", "therefore"]}}],
        [{"LOWER": "because"}, {"LOWER": "of"}],
        [{"LOWER": "as"}, {"LOWER": "a"}, {"LOWER": "result"}],
    ],
    "evidentials": [
        [{"LOWER": "according"}, {"LOWER": "to"}],
        [{"POS": "PROPN"},
         {"LOWER": {"IN": ["states", "argues", "claims", "notes"]}},
         {"LOWER": "that"}],
        [{"LOWER": {"IN": ["is", "was", "are", "were"]}},
         {"LOWER": {"IN": ["cited", "reported", "claimed", "noted"]}}],
    ],
    "self_mentions_organizational": [
        # Text organization and structure
        [{"LOWER": {"IN": ["i", "we"]}}, 
         {"LOWER": {"IN": ["discuss", "present", "describe", "introduce", "outline", "review", 
                         "summarize", "analyze", "examine", "consider", "focus", "begin", 
                         "conclude", "turn to", "return to", "proceed"]}}],
        [{"LOWER": {"IN": ["in", "throughout", "within"]}},
         {"LOWER": {"IN": ["this", "the", "my", "our"]}},
         {"LOWER": {"IN": ["paper", "article", "study", "section", "chapter", "analysis"]}}],
    ],
    "self_mentions_stance": [
        # Expressing stance/opinion
        [{"LOWER": {"IN": ["i", "we"]}},
         {"LOWER": {"IN": ["argue", "claim", "suggest", "propose", "believe", "think", 
                         "feel", "agree", "disagree", "acknowledge", "contend", 
                         "hypothesize", "speculate", "assume"]}}],
        [{"LOWER": {"IN": ["in", "from"]}},
         {"LOWER": {"IN": ["my", "our"]}},
         {"LOWER": {"IN": ["view", "perspective", "opinion", "judgment", "understanding"]}}],
    ],
    "self_mentions_methodological": [
        # Research methodology
        [{"LOWER": {"IN": ["i", "we"]}},
         {"LOWER": {"IN": ["collected", "analyzed", "measured", "calculated", "observed", 
                         "conducted", "surveyed", "interviewed", "sampled", "recruited", 
                         "selected", "designed", "developed", "created", "used", "applied"]}}],
        [{"LOWER": {"IN": ["my", "our"]}},
         {"LOWER": {"IN": ["method", "approach", "methodology", "analysis", "procedure", 
                         "experiment", "sample", "dataset", "participants"]}}],
    ],
}


In [ ]:
# Section 3: Registering the spaCy Components

@Language.component("metadiscourse_detector")
def metadiscourse_detector(doc):
    """Improved metadiscourse detector with contextual filtering."""
    if not Doc.has_extension("metadiscourse_markers"):
        Doc.set_extension("metadiscourse_markers", default={})
    
    results = {
        "transitions": [],
        "frame_markers": [],
        "endophoric_markers": [],
        "evidentials": [],
        "code_glosses": [],
        "hedges": [],
        "boosters": [],
        "attitude_markers": [],
        "engagement_markers": [],
        "self_mentions": []
    }
    
    # Use PhraseMatcher for exact phrase matches.
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    for category, markers in INTERACTIVE_MARKERS.items():
        patterns = [nlp.make_doc(marker) for marker in markers]
        matcher.add(category, patterns)
    for category, markers in INTERACTIONAL_MARKERS.items():
        patterns = [nlp.make_doc(marker) for marker in markers]
        matcher.add(category, patterns)
    
    matches = matcher(doc)
    for match_id, start, end in matches:
        category = doc.vocab.strings[match_id]
        span = doc[start:end]
        results[category].append((span.text, (start, end)))
    
    # Use Matcher for contextual filtering.
    context_matcher = Matcher(nlp.vocab)
    for category, patterns in CONTEXT_PATTERNS.items():
        # Skip self-mention sub-categories as they'll be handled separately
        if category.startswith("self_mentions_"):
            continue
        
        for i, pattern_rule in enumerate(patterns):
            context_matcher.add(f"{category}_{i}", [pattern_rule])
    
    context_matches = context_matcher(doc)
    for match_id, start, end in context_matches:
        match_name = doc.vocab.strings[match_id]
        category_base = match_name.split('_')[0]
        category_map = {
            'transition': 'transitions',
            'frame': 'frame_markers',
            'endophoric': 'endophoric_markers',
            'code': 'code_glosses',
            'attitude': 'attitude_markers',
            'engagement': 'engagement_markers',
            'self': 'self_mentions',
            'hedge': 'hedges',
            'booster': 'boosters',
            'evidential': 'evidentials'
        }
        
        category = category_map.get(category_base, category_base)
        if category in results:
            span = doc[start:end]
            already_captured = any(start <= s < end or start < e <= end for _, (s, e) in results[category])
            if not already_captured:
                results[category].append((span.text, (start, end)))
    
    # Additional detection for citations
    citation_pattern = re.compile(r'\(\s*[A-Za-z]+\s*,\s*\d{4}\s*\)')
    for match in citation_pattern.finditer(doc.text):
        start_char = match.start()
        end_char = match.end()
        start_token = None
        end_token = None
        
        # Find token span that corresponds to the character span
        for i, token in enumerate(doc):
            if token.idx <= start_char and token.idx + len(token.text) > start_char:
                start_token = i
            if token.idx < end_char and token.idx + len(token.text) >= end_char:
                end_token = i + 1
                break
        
        if start_token is not None and end_token is not None:
            span_text = doc[start_token:end_token].text
            results["evidentials"].append((span_text, (start_token, end_token)))
    
    doc._.metadiscourse_markers = results
    return doc

@Language.component("self_mention_categorizer")
def self_mention_categorizer(doc):
    """Categorize self-mentions into sub-categories based on context."""
    if not Doc.has_extension("self_mention_categories"):
        Doc.set_extension("self_mention_categories", default={
            "organizational": [],
            "stance": [],
            "methodological": [],
            "other": []  # Uncategorized self-mentions
        })
    
    # Get the basic self-mentions detected by the main metadiscourse detector
    if not hasattr(doc._, "metadiscourse_markers") or "self_mentions" not in doc._.metadiscourse_markers:
        doc._.self_mention_categories = {
            "organizational": [],
            "stance": [],
            "methodological": [],
            "other": []
        }
        return doc
    
    # Extract all detected self-mentions
    all_self_mentions = doc._.metadiscourse_markers["self_mentions"]
    
    # Convert to a set of spans for quick membership testing
    self_mention_spans = set()
    for text, (start, end) in all_self_mentions:
        for i in range(start, end):
            self_mention_spans.add(i)
    
    # Initialize sub-categories
    organizational = []
    stance = []
    methodological = []
    other = list(all_self_mentions)  # Start with all mentions, will remove as we categorize
    
    # Use matcher for contextual detection
    matcher = Matcher(nlp.vocab)
    
    # Add organizational patterns
    for i, pattern in enumerate(CONTEXT_PATTERNS.get("self_mentions_organizational", [])):
        matcher.add(f"organizational_{i}", [pattern])
    
    # Add stance patterns
    for i, pattern in enumerate(CONTEXT_PATTERNS.get("self_mentions_stance", [])):
        matcher.add(f"stance_{i}", [pattern])
    
    # Add methodological patterns
    for i, pattern in enumerate(CONTEXT_PATTERNS.get("self_mentions_methodological", [])):
        matcher.add(f"methodological_{i}", [pattern])
    
    # Get matches
    matches = matcher(doc)
    processed_spans = set()
    
    for match_id, start, end in matches:
        match_name = doc.vocab.strings[match_id]
        category_base = match_name.split('_')[0]
        span_text = doc[start:end].text
        
        # Check if this span contains a self-mention
        contains_self_mention = False
        for i in range(start, end):
            if i in self_mention_spans:
                contains_self_mention = True
                break
        
        if not contains_self_mention:
            continue
        
        # Create a unique span identifier
        span_key = (start, end, span_text)
        if span_key in processed_spans:
            continue
        processed_spans.add(span_key)
        
        # Add to appropriate category and remove from 'other'
        if category_base == "organizational":
            organizational.append((span_text, (start, end)))
            # Remove overlapping spans from 'other'
            other = [(t, (s, e)) for t, (s, e) in other 
                     if not (s >= start and s < end) and not (e > start and e <= end)]
        elif category_base == "stance":
            stance.append((span_text, (start, end)))
            other = [(t, (s, e)) for t, (s, e) in other 
                     if not (s >= start and s < end) and not (e > start and e <= end)]
        elif category_base == "methodological":
            methodological.append((span_text, (start, end)))
            other = [(t, (s, e)) for t, (s, e) in other 
                     if not (s >= start and s < end) and not (e > start and e <= end)]
    
    # Store results
    doc._.self_mention_categories = {
        "organizational": organizational,
        "stance": stance,
        "methodological": methodological,
        "other": other
    }
    return doc

# Register components after they're defined
if not Language.has_factory("metadiscourse_detector"):
    Language.factory("metadiscourse_detector")(metadiscourse_detector)

if not Language.has_factory("self_mention_categorizer"):
    Language.factory("self_mention_categorizer")(self_mention_categorizer)

# Add the components to the pipeline
if "metadiscourse_detector" not in nlp.pipe_names:
    nlp.add_pipe("metadiscourse_detector", last=True)
    
if "self_mention_categorizer" not in nlp.pipe_names:
    nlp.add_pipe("self_mention_categorizer", last=True)

print("NLP pipeline components registered successfully.")

In [ ]:
# Section 4: Analysis Functions

def analyze_text(text, nlp=None):
    """Analyze a text for metadiscourse markers and calculate statistics."""
    if nlp is None:
        nlp = spacy.load("en_core_web_trf")
        if "metadiscourse_detector" not in nlp.pipe_names:
            nlp.add_pipe("metadiscourse_detector", last=True)
        if "self_mention_categorizer" not in nlp.pipe_names:
            nlp.add_pipe("self_mention_categorizer", last=True)
    
    doc = nlp(text)
    word_count = len([token for token in doc if not token.is_punct and not token.is_space])
    markers = doc._.metadiscourse_markers
    counts = {category: len(instances) for category, instances in markers.items()}
    
    # Add self-mention sub-categories counts
    if hasattr(doc._, "self_mention_categories"):
        self_mention_categories = doc._.self_mention_categories
        counts["self_mentions_organizational"] = len(self_mention_categories["organizational"])
        counts["self_mentions_stance"] = len(self_mention_categories["stance"])
        counts["self_mentions_methodological"] = len(self_mention_categories["methodological"])
        counts["self_mentions_other"] = len(self_mention_categories["other"])
    
    interactive_categories = ["transitions", "frame_markers", "endophoric_markers", "evidentials", "code_glosses"]
    interactional_categories = ["hedges", "boosters", "attitude_markers", "engagement_markers", "self_mentions"]
    
    interactive_total = sum(counts.get(cat, 0) for cat in interactive_categories)
    interactional_total = sum(counts.get(cat, 0) for cat in interactional_categories)
    density_factor = 1000 / word_count if word_count > 0 else 0
    
    results = {
        "interactive_total": interactive_total,
        "interactional_total": interactional_total,
        "interactive_density": interactive_total * density_factor,
        "interactional_density": interactional_total * density_factor,
        "word_count": word_count
    }
    
    # Standard categories density
    for category in interactive_categories + interactional_categories:
        results[category] = counts.get(category, 0)
        results[f"{category}_density"] = counts.get(category, 0) * density_factor
    
    # Add self-mention sub-categories density
    for category in ["self_mentions_organizational", "self_mentions_stance", 
                    "self_mentions_methodological", "self_mentions_other"]:
        if category in counts:
            results[category] = counts[category]
            results[f"{category}_density"] = counts[category] * density_factor
    
    return results

def load_corpus(corpus_path, language_map=None):
    """Load corpus texts and analyze them for metadiscourse markers."""
    results = []
    try:
        nlp = spacy.load("en_core_web_trf")
    except Exception as e:
        nlp = spacy.load("en_core_web_sm")
        print("Warning: Using a smaller spaCy model. For better results, install en_core_web_trf.")
    
    if "metadiscourse_detector" not in nlp.pipe_names:
        nlp.add_pipe("metadiscourse_detector", last=True)
    
    if "self_mention_categorizer" not in nlp.pipe_names:
        nlp.add_pipe("self_mention_categorizer", last=True)
    
    for file_path in os.listdir(corpus_path):
        if file_path.endswith(".txt"):
            full_path = os.path.join(corpus_path, file_path)
            print(f"Processing {file_path}...")
            with open(full_path, 'r', encoding='utf-8') as f:
                text = f.read()
            native_language = language_map.get(file_path, "Unknown") if language_map else "Unknown"
            analysis_result = analyze_text(text, nlp)
            analysis_result["file_name"] = file_path
            analysis_result["Native_Language"] = native_language
            results.append(analysis_result)
    
    return pd.DataFrame(results)

def load_language_map(file_path):
    """Load mapping between filenames and native languages."""
    language_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 2:
                filename, language = parts[0].strip(), parts[1].strip()
                language_map[filename] = language
    return language_map

In [ ]:
# Section 5: LaTeX Table Generation and Statistical Testing Functions

def generate_latex_tables(df, output_dir='results'):
    """Generate enhanced LaTeX tables for academic publication."""
    os.makedirs(output_dir, exist_ok=True)
    
    print("Generating language summary table...")
    if 'Native_Language' in df.columns:
        language_summary = df.groupby('Native_Language').agg({
            'interactive_density': ['mean', 'std', 'count'],
            'interactional_density': ['mean', 'std', 'count'],
            'word_count': ['mean', 'std']
        })
        language_summary_latex = language_summary.to_latex(
            float_format="{:.2f}".format,
            caption="Summary of Metadiscourse Markers by Native Language",
            label="tab:language_summary",
            position="h!",
            multirow=True
        )
        language_summary_latex = language_summary_latex.replace(
            "\\end{tabular}",
            "\\bottomrule\n\\end{tabular}\n\\caption*{Note: Values represent mean and standard deviation per 1000 words.}"
        )
        with open(os.path.join(output_dir, 'language_summary_table_full.tex'), 'w') as f:
            f.write(language_summary_latex)
        print(f"Language summary table saved to {os.path.join(output_dir, 'language_summary_table_full.tex')}")
    
    print("Generating detailed marker statistics table...")
    interactive_cols = [col for col in df.columns if col.endswith('_density') and 
                        any(col.startswith(cat) for cat in ['transitions', 'frame_markers', 
                                                          'endophoric_markers', 'evidentials', 'code_glosses'])]
    if interactive_cols:
        interactive_stats = df[interactive_cols].describe().T.reset_index()
        interactive_stats.columns = ['Marker', 'Count', 'Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max']
        interactive_latex = interactive_stats.to_latex(
            index=False,
            float_format="{:.3f}".format,
            caption="Descriptive Statistics for Interactive Metadiscourse Markers",
            label="tab:interactive_markers",
            position="h!"
        )
        # Fix the tabular environment formatting
        interactive_latex = interactive_latex.replace(
            "\\begin{tabular}{lrrrrrrrr}",
            "\\begin{tabular}{lrrrrrrr}"
        ).replace(
            "\\bottomrule\n\\bottomrule",
            "\\bottomrule"
        ).replace(
            "\\end{tabular}",
            "\\end{tabular}\n\\caption*{Note: Values represent frequency per 1000 words.}"
        )
        with open(os.path.join(output_dir, 'interactive_markers_table_full.tex'), 'w') as f:
            f.write(interactive_latex)
        print(f"Interactive markers table saved to {os.path.join(output_dir, 'interactive_markers_table_full.tex')}")
    
    interactional_cols = [col for col in df.columns if col.endswith('_density') and 
                          any(col.startswith(cat) for cat in ['hedges', 'boosters', 'attitude_markers', 
                                                           'engagement_markers', 'self_mentions']) and 
                          not any(cat in col for cat in ['organizational', 'stance', 'methodological', 'other'])]
    if interactional_cols:
        interactional_stats = df[interactional_cols].describe().T.reset_index()
        interactional_stats.columns = ['Marker', 'Count', 'Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max']
        interactional_latex = interactional_stats.to_latex(
            index=False,
            float_format="{:.3f}".format,
            caption="Descriptive Statistics for Interactional Metadiscourse Markers",
            label="tab:interactional_markers",
            position="h!"
        )
        # Fix the tabular environment formatting
        interactional_latex = interactional_latex.replace(
            "\\begin{tabular}{lrrrrrrrr}",
            "\\begin{tabular}{lrrrrrrr}"
        ).replace(
            "\\bottomrule\n\\bottomrule",
            "\\bottomrule"
        ).replace(
            "\\end{tabular}",
            "\\end{tabular}\n\\caption*{Note: Values represent frequency per 1000 words.}"
        )
        with open(os.path.join(output_dir, 'interactional_markers_table_full.tex'), 'w') as f:
            f.write(interactional_latex)
        print(f"Interactional markers table saved to {os.path.join(output_dir, 'interactional_markers_table_full.tex')}")
    
    # Add a new table for self-mention sub-categories
    self_mention_cols = [col for col in df.columns if col.endswith('_density') and 
                         any(cat in col for cat in ['self_mentions_organizational', 
                                                   'self_mentions_stance', 
                                                   'self_mentions_methodological', 
                                                   'self_mentions_other'])]
    if self_mention_cols:
        self_mention_stats = df[self_mention_cols].describe().T.reset_index()
        self_mention_stats.columns = ['Marker', 'Count', 'Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max']
        # Clean up the marker names for display
        self_mention_stats['Marker'] = self_mention_stats['Marker'].str.replace('_density', '')
        self_mention_latex = self_mention_stats.to_latex(
            index=False,
            float_format="{:.3f}".format,
            caption="Descriptive Statistics for Self-Mention Sub-Categories",
            label="tab:self_mention_categories",
            position="h!"
        )
        # Fix the tabular environment formatting
        self_mention_latex = self_mention_latex.replace(
            "\\begin{tabular}{lrrrrrrrr}",
            "\\begin{tabular}{lrrrrrrr}"
        ).replace(
            "\\bottomrule\n\\bottomrule",
            "\\bottomrule"
        ).replace(
            "\\end{tabular}",
            "\\end{tabular}\n\\caption*{Note: Values represent frequency per 1000 words.}"
        )
        with open(os.path.join(output_dir, 'self_mention_categories_table.tex'), 'w') as f:
            f.write(self_mention_latex)
        print(f"Self-mention categories table saved to {os.path.join(output_dir, 'self_mention_categories_table.tex')}")
    
    print("Generating correlation matrix table...")
    # For correlation matrix, use the main categories (without self-mention sub-categories)
    main_density_cols = [col for col in df.columns if col.endswith('_density') and
                         not any(sub in col for sub in ['organizational', 'stance', 'methodological', 'other'])]
    if main_density_cols:
        corr_matrix = df[main_density_cols].corr()
        corr_latex = corr_matrix.to_latex(
            float_format="{:.3f}".format,
            caption="Correlation Matrix of Metadiscourse Markers",
            label="tab:correlation_matrix",
            position="h!"
        )
        corr_latex = corr_latex.replace(
            "\\end{tabular}",
            "\\bottomrule\n\\end{tabular}\n\\caption*{Note: Values represent Pearson correlation coefficients.}"
        )
        with open(os.path.join(output_dir, 'correlation_matrix_full.tex'), 'w') as f:
            f.write(corr_latex)
        print(f"Correlation matrix table saved to {os.path.join(output_dir, 'correlation_matrix_full.tex')}")
    
    print("Creating combined LaTeX file with all tables...")
    combined_latex = (
        "\\documentclass{article}\n"
        "\\usepackage{booktabs}\n"
        "\\usepackage{caption}\n"
        "\\usepackage{multirow}\n"
        "\\usepackage{float}\n"
        "\\begin{document}\n"
        "\\title{Metadiscourse Analysis Results}\n"
        "\\author{Metadiscourse Analyzer}\n"
        "\\date{\\today}\n"
        "\\maketitle\n"
        "\\section{Summary Statistics}\n"
    )
    if 'Native_Language' in df.columns:
        combined_latex += (
            "\\subsection{Language Group Summary}\n"
            "\\input{" + os.path.join(output_dir, 'language_summary_table_full').replace('\\', '/') + "}\n"
        )
    combined_latex += (
        "\\subsection{Metadiscourse Marker Statistics}\n"
        "\\input{" + os.path.join(output_dir, 'interactive_markers_table_full').replace('\\', '/') + "}\n"
        "\\input{" + os.path.join(output_dir, 'interactional_markers_table_full').replace('\\', '/') + "}\n"
    )
    if os.path.exists(os.path.join(output_dir, 'self_mention_categories_table.tex')):
        combined_latex += (
            "\\subsection{Self-Mention Sub-Categories}\n"
            "\\input{" + os.path.join(output_dir, 'self_mention_categories_table').replace('\\', '/') + "}\n"
        )
    combined_latex += (
        "\\subsection{Correlation Analysis}\n"
        "\\input{" + os.path.join(output_dir, 'correlation_matrix_full').replace('\\', '/') + "}\n"
        "\\end{document}\n"
    )
    with open(os.path.join(output_dir, 'metadiscourse_analysis_tables.tex'), 'w') as f:
        f.write(combined_latex)

def perform_language_group_statistics(df, output_dir='results'):
    """Perform statistical tests to compare language groups."""
    os.makedirs(output_dir, exist_ok=True)
    if 'Native_Language' not in df.columns or len(df['Native_Language'].unique()) <= 1:
        return
    
    # Measures to test (using density keys) - include self-mention sub-categories
    measures_to_test = [
        'interactional_density',
        'interactive_density',
        'transitions_density',
        'frame_markers_density',
        'endophoric_markers_density',
        'evidentials_density',
        'code_glosses_density',
        'hedges_density',
        'boosters_density',
        'attitude_markers_density',
        'engagement_markers_density',
        'self_mentions_density'
    ]
    
    # Include self-mention sub-categories if available
    self_mention_measures = [
        'self_mentions_organizational_density',
        'self_mentions_stance_density',
        'self_mentions_methodological_density',
        'self_mentions_other_density'
    ]
    
    for measure in self_mention_measures:
        if measure in df.columns:
            measures_to_test.append(measure)
    
    results = []
    for measure in measures_to_test:
        if measure in df.columns:
            try:
                formula = f"{measure} ~ C(Native_Language)"
                model = ols(formula, data=df).fit()
                anova_table = sm.stats.anova_lm(model, typ=2)
                f_value = anova_table.iloc[0]['F']
                p_value = anova_table.iloc[0]['PR(>F)']
                posthoc_results = None
                if p_value < 0.05:
                    mc = MultiComparison(df[measure], df['Native_Language'])
                    posthoc_results = mc.tukeyhsd()
                    posthoc_file = os.path.join(output_dir, f"{measure}_posthoc.txt")
                    with open(posthoc_file, 'w') as f:
                        f.write(str(posthoc_results))
                results.append({
                    'Measure': measure,
                    'F_value': f_value,
                    'p_value': p_value,
                    'Significant': p_value < 0.05,
                    'Posthoc_file': f"{measure}_posthoc.txt" if p_value < 0.05 else None
                })
            except Exception as e:
                print(f"Error performing ANOVA for {measure}: {e}")
    
    if results:
        results_df = pd.DataFrame(results)
        results_latex = results_df.to_latex(
            index=False,
            float_format="{:.3f}".format,
            caption="ANOVA Results for Language Group Differences",
            label="tab:anova_results",
            position="h!"
        )
        results_latex = results_latex.replace(
            "\\end{tabular}",
            "\\bottomrule\n\\end{tabular}\n\\caption*{Note: * indicates significant at p < 0.05.}"
        )
        with open(os.path.join(output_dir, 'anova_results.tex'), 'w') as f:
            f.write(results_latex)

In [ ]:
# Section 6: Visualization Functions

def generate_visualizations(df, output_dir='results'):
    """Generate improved visualizations for metadiscourse analysis with enhanced self-mention categories."""
    os.makedirs(output_dir, exist_ok=True)
    sns.set(style="whitegrid")
    plt.rcParams.update({
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'figure.figsize': (12, 8)
    })
    
    # 1. Scatter Plot: Interactive vs. Interactional Markers with Regression Line
    print("Generating interactive vs. interactional markers scatter plot...")
    plt.figure(figsize=(10, 8))
    if 'Native_Language' in df.columns:
        sns.scatterplot(
            data=df,
            x='interactive_density',
            y='interactional_density',
            hue='Native_Language',
            s=100,
            alpha=0.7
        )
    else:
        sns.scatterplot(
            data=df,
            x='interactive_density',
            y='interactional_density',
            s=100,
            alpha=0.7
        )
    plt.title('Interactive vs. Interactional Metadiscourse Markers')
    plt.xlabel('Interactive Markers (per 1000 words)')
    plt.ylabel('Interactional Markers (per 1000 words)')
    plt.grid(True, linestyle='--', alpha=0.7)
    sns.regplot(
        x='interactive_density',
        y='interactional_density',
        data=df,
        scatter=False,
        ci=None,
        line_kws={"color": "red", "lw": 2, "linestyle": "--"}
    )
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'interactive_vs_interactional.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Boxplot of Main Marker Categories
    print("Generating boxplots of marker categories...")
    main_density_cols = [col for col in df.columns if col.endswith('_density') and 
                        not col.startswith('interactive_') and 
                        not col.startswith('interactional_') and
                        not 'self_mentions_' in col]  # Exclude sub-categories
    
    if main_density_cols:
        id_vars = []
        if 'Native_Language' in df.columns:
            id_vars.append('Native_Language')
        melted_df = pd.melt(
            df,
            id_vars=id_vars,
            value_vars=main_density_cols,
            var_name='Marker_Category',
            value_name='Density'
        )
        melted_df['Marker_Category'] = melted_df['Marker_Category'].str.replace('_density', '')
        plt.figure(figsize=(14, 8))
        sns.boxplot(
            x='Marker_Category',
            y='Density',
            data=melted_df,
            hue=None,
            palette="Set3"
        )
        plt.xticks(rotation=45, ha='right')
        plt.title('Distribution of Metadiscourse Marker Categories')
        plt.xlabel('Marker Category')
        plt.ylabel('Density (per 1000 words)')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'marker_categories_boxplot.png'), dpi=300, bbox_inches='tight')
        plt.close()
    
    # 3. Self-Mention Sub-Categories Boxplot
    self_mention_cols = [col for col in df.columns if col.endswith('_density') and 
                         any(sub in col for sub in ['self_mentions_organizational', 
                                                  'self_mentions_stance', 
                                                  'self_mentions_methodological', 
                                                  'self_mentions_other'])]
    
    if self_mention_cols and len(self_mention_cols) > 1:
        id_vars = []
        if 'Native_Language' in df.columns:
            id_vars.append('Native_Language')
        
        melted_df = pd.melt(
            df,
            id_vars=id_vars,
            value_vars=self_mention_cols,
            var_name='Self_Mention_Type',
            value_name='Density'
        )
        melted_df['Self_Mention_Type'] = melted_df['Self_Mention_Type'].str.replace('_density', '').str.replace('self_mentions_', '')
        
        plt.figure(figsize=(12, 8))
        sns.boxplot(
            x='Self_Mention_Type',
            y='Density',
            data=melted_df,
            palette="Set2"
        )
        plt.title('Distribution of Self-Mention Sub-Categories')
        plt.xlabel('Self-Mention Function')
        plt.ylabel('Density (per 1000 words)')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'self_mention_categories_boxplot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # If we have language information, create a stacked bar chart of self-mention categories by language
        if 'Native_Language' in df.columns:
            plt.figure(figsize=(14, 8))
            
            # Calculate means by language group
            pivot_df = pd.pivot_table(
                melted_df, 
                values='Density', 
                index='Native_Language',
                columns='Self_Mention_Type', 
                aggfunc='mean'
            )
            
            pivot_df.plot(kind='bar', stacked=True, figsize=(14, 8), colormap='viridis')
            plt.title('Self-Mention Functions by Native Language')
            plt.xlabel('Native Language')
            plt.ylabel('Density (per 1000 words)')
            plt.legend(title='Function Type')
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'self_mention_by_language.png'), dpi=300, bbox_inches='tight')
            plt.close()
    
    # 4. Language Group Comparisons: Bar Plots and Heatmap
    if 'Native_Language' in df.columns:
        # 4.1 Interactive markers by language
        plt.figure(figsize=(12, 6))
        bar_plot = sns.barplot(
            x='Native_Language',
            y='interactive_density',
            data=df,
            palette="viridis",
            errorbar=('ci', 95),
            capsize=0.2
        )
        plt.title('Interactive Metadiscourse Markers by Native Language')
        plt.xlabel('Native Language')
        plt.ylabel('Interactive Markers (per 1000 words)')
        plt.grid(True, linestyle='--', alpha=0.7)
        for bar in bar_plot.patches:
            bar_plot.text(
                bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.3,
                f'{bar.get_height():.1f}',
                ha='center',
                va='bottom',
                fontsize=10
            )
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'interactive_by_language.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # 4.2 Interactional markers by language
        plt.figure(figsize=(12, 6))
        bar_plot = sns.barplot(
            x='Native_Language',
            y='interactional_density',
            data=df,
            palette="plasma",
            errorbar=('ci', 95),
            capsize=0.2
        )
        plt.title('Interactional Metadiscourse Markers by Native Language')
        plt.xlabel('Native Language')
        plt.ylabel('Interactional Markers (per 1000 words)')
        plt.grid(True, linestyle='--', alpha=0.7)
        for bar in bar_plot.patches:
            bar_plot.text(
                bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.3,
                f'{bar.get_height():.1f}',
                ha='center',
                va='bottom',
                fontsize=10
            )
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'interactional_by_language.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # 4.3 Heatmap of marker categories by language
        if main_density_cols:
            heatmap_data = df.groupby('Native_Language')[main_density_cols].mean()
            heatmap_data.columns = [col.replace('_density', '') for col in heatmap_data.columns]
            plt.figure(figsize=(16, 10))
            sns.heatmap(
                heatmap_data,
                annot=True,
                fmt='.2f',
                cmap='YlGnBu',
                linewidths=0.5,
                cbar_kws={'label': 'Density (per 1000 words)'}
            )
            plt.title('Metadiscourse Marker Categories by Native Language')
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'marker_categories_heatmap.png'), dpi=300, bbox_inches='tight')
            plt.close()
    
    # 5. Self-Mention Functions Pie Chart
    if all(col in df.columns for col in ['self_mentions_organizational', 'self_mentions_stance', 
                                        'self_mentions_methodological', 'self_mentions_other']):
        total_org = df['self_mentions_organizational'].sum()
        total_stance = df['self_mentions_stance'].sum()
        total_method = df['self_mentions_methodological'].sum()
        total_other = df['self_mentions_other'].sum()
        
        total_all = total_org + total_stance + total_method + total_other
        
        if total_all > 0:  # Ensure we have some self-mentions
            labels = ['Organizational', 'Stance', 'Methodological', 'Other']
            sizes = [total_org, total_stance, total_method, total_other]
            explode = (0.1, 0.1, 0.1, 0.1)  # explode all slices for visibility
            
            plt.figure(figsize=(10, 10))
            plt.pie(sizes, explode=explode, labels=labels, autopct='%1.1f%%',
                    shadow=True, startangle=140, colors=['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3'])
            plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
            plt.title('Distribution of Self-Mention Functions', fontsize=16)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'self_mention_functions_pie.png'), dpi=300)
            plt.close()
    
    # 6. Histogram of Word Counts
    print("Generating histogram of word counts...")
    plt.figure(figsize=(10, 6))
    sns.histplot(df['word_count'], bins=30, color='skyblue', kde=True)
    plt.title("Distribution of Word Counts")
    plt.xlabel("Word Count")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'word_count_histogram.png'), dpi=300)
    plt.close()
    
    print(f"Visualizations saved to {output_dir}/")

In [ ]:
# Section 7: Additional Analysis Functions

def analyze_metadiscourse_distribution(df, output_dir='results'):
    """Analyze the distribution of metadiscourse markers and generate distribution data."""
    os.makedirs(output_dir, exist_ok=True)
    interactive_categories = ["transitions", "frame_markers", "endophoric_markers", "evidentials", "code_glosses"]
    interactional_categories = ["hedges", "boosters", "attitude_markers", "engagement_markers", "self_mentions"]
    
    # Self-mention sub-categories for additional analysis
    self_mention_subcategories = ["self_mentions_organizational", "self_mentions_stance", 
                                 "self_mentions_methodological", "self_mentions_other"]
    
    # Main categories distribution
    total_markers = {}
    for category in interactive_categories + interactional_categories:
        if category in df.columns:
            total_markers[category] = df[category].sum()
    
    total_count = sum(total_markers.values())
    percentages = {category: (count / total_count) * 100 for category, count in total_markers.items()} if total_count > 0 else {}
    
    distribution_df = pd.DataFrame({
        'Category': list(percentages.keys()),
        'Count': list(total_markers.values()),
        'Percentage': list(percentages.values())
    })
    
    distribution_df['Type'] = distribution_df['Category'].apply(
        lambda x: 'Interactive' if x in interactive_categories else 'Interactional'
    )
    
    distribution_df = distribution_df.sort_values('Percentage', ascending=False)
    distribution_df.to_csv(os.path.join(output_dir, 'metadiscourse_distribution.csv'), index=False)
    
    # Self-mention sub-categories distribution
    self_mention_distribution = {}
    for category in self_mention_subcategories:
        if category in df.columns:
            self_mention_distribution[category] = df[category].sum()
    
    self_mention_total = sum(self_mention_distribution.values())
    self_mention_percentages = {cat: (count / self_mention_total) * 100 for cat, count in self_mention_distribution.items()} if self_mention_total > 0 else {}
    
    if self_mention_percentages:
        self_mention_df = pd.DataFrame({
            'Category': list(self_mention_percentages.keys()),
            'Count': list(self_mention_distribution.values()),
            'Percentage': list(self_mention_percentages.values())
        })
        
        self_mention_df['Category'] = self_mention_df['Category'].str.replace('self_mentions_', '').str.capitalize()
        self_mention_df = self_mention_df.sort_values('Percentage', ascending=False)
        self_mention_df.to_csv(os.path.join(output_dir, 'self_mention_distribution.csv'), index=False)
    
    # Return summary statistics
    result = {
        'interactive_percentage': (sum(total_markers.get(cat, 0) for cat in interactive_categories) / total_count * 100) if total_count > 0 else 0,
        'interactional_percentage': (sum(total_markers.get(cat, 0) for cat in interactional_categories) / total_count * 100) if total_count > 0 else 0,
        'most_frequent_category': distribution_df.iloc[0]['Category'] if not distribution_df.empty else None,
        'most_frequent_percentage': distribution_df.iloc[0]['Percentage'] if not distribution_df.empty else 0,
        'least_frequent_category': distribution_df.iloc[-1]['Category'] if not distribution_df.empty else None,
        'least_frequent_percentage': distribution_df.iloc[-1]['Percentage'] if not distribution_df.empty else 0
    }
    
    # Add self-mention sub-category stats if available
    if self_mention_percentages:
        result['self_mentions_organizational_pct'] = self_mention_percentages.get('self_mentions_organizational', 0)
        result['self_mentions_stance_pct'] = self_mention_percentages.get('self_mentions_stance', 0)
        result['self_mentions_methodological_pct'] = self_mention_percentages.get('self_mentions_methodological', 0)
        result['self_mentions_other_pct'] = self_mention_percentages.get('self_mentions_other', 0)
    
    return result

def calculate_shannon_entropy(df, output_dir='results'):
    """Calculate Shannon entropy for metadiscourse markers to measure diversity."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Main categories for entropy calculation
    categories = [
        "transitions", "frame_markers", "endophoric_markers", "evidentials", "code_glosses",
        "hedges", "boosters", "attitude_markers", "engagement_markers", "self_mentions"
    ]
    
    # Only include categories present in the data
    categories = [cat for cat in categories if cat in df.columns]
    
    if not categories:
        print("No marker categories found in the dataframe.")
        return {}
    
    # Calculate entropy for main categories
    entropy_results = []
    for idx, row in df.iterrows():
        counts = [row[cat] for cat in categories]
        total = sum(counts)
        probabilities = [count / total if total > 0 else 0 for count in counts]
        entropy = -sum(p * math.log2(p) if p > 0 else 0 for p in probabilities)
        max_entropy = math.log2(len(categories))
        normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
        
        result = {
            'file_name': row['file_name'] if 'file_name' in row else f"Text_{idx}",
            'Shannon_Entropy': entropy,
            'Normalized_Entropy': normalized_entropy,
            'Max_Entropy': max_entropy
        }
        
        # Calculate entropy for self-mention sub-categories if available
        self_mention_cats = ["self_mentions_organizational", "self_mentions_stance", 
                            "self_mentions_methodological", "self_mentions_other"]
        
        if all(cat in row for cat in self_mention_cats):
            sm_counts = [row[cat] for cat in self_mention_cats]
            sm_total = sum(sm_counts)
            
            if sm_total > 0:
                sm_probabilities = [count / sm_total for count in sm_counts]
                sm_entropy = -sum(p * math.log2(p) if p > 0 else 0 for p in sm_probabilities)
                sm_max_entropy = math.log2(len(self_mention_cats))
                sm_normalized_entropy = sm_entropy / sm_max_entropy if sm_max_entropy > 0 else 0
                
                result['Self_Mentions_Entropy'] = sm_entropy
                result['Self_Mentions_Normalized_Entropy'] = sm_normalized_entropy
        
        if 'Native_Language' in row:
            result['Native_Language'] = row['Native_Language']
            
        entropy_results.append(result)
    
    entropy_df = pd.DataFrame(entropy_results)
    entropy_df.to_csv(os.path.join(output_dir, 'entropy_analysis.csv'), index=False)
    
    # Prepare summary statistics for return
    summary = {
        'mean_entropy': entropy_df['Shannon_Entropy'].mean(),
        'mean_normalized_entropy': entropy_df['Normalized_Entropy'].mean(),
        'max_entropy': entropy_df['Max_Entropy'].iloc[0] if not entropy_df.empty else 0,
        'min_normalized_entropy': entropy_df['Normalized_Entropy'].min(),
        'max_normalized_entropy': entropy_df['Normalized_Entropy'].max()
    }
    
    # Add self-mention entropy stats if available
    if 'Self_Mentions_Normalized_Entropy' in entropy_df.columns:
        summary['self_mentions_mean_entropy'] = entropy_df['Self_Mentions_Entropy'].mean()
        summary['self_mentions_normalized_entropy'] = entropy_df['Self_Mentions_Normalized_Entropy'].mean()
    
    # Add language-specific stats if available
    if 'Native_Language' in entropy_df.columns:
        language_stats = entropy_df.groupby('Native_Language')['Normalized_Entropy'].agg(['mean', 'std', 'min', 'max'])
        summary['language_stats'] = language_stats.to_dict(orient='index')
        
        # ANOVA test for language differences in entropy
        try:
            formula = "Normalized_Entropy ~ C(Native_Language)"
            model = ols(formula, data=entropy_df).fit()
            anova_table = sm.stats.anova_lm(model, typ=2)
            summary['entropy_anova_f'] = anova_table.iloc[0]['F']
            summary['entropy_anova_p'] = anova_table.iloc[0]['PR(>F)']
            summary['entropy_significant_diff'] = anova_table.iloc[0]['PR(>F)'] < 0.05
        except Exception as e:
            print(f"Error performing ANOVA for entropy: {e}")
    
    return summary

def save_parsed_output(results_df, output_dir='intermediate_data'):
    """Save parsed metadiscourse data in a format optimized for Bayesian analysis.
    
    This function creates a structured dataset with all necessary information
    for running Bayesian regression models on metadiscourse markers.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Save the complete processed dataframe
    results_df.to_csv(os.path.join(output_dir, 'complete_metadiscourse_data.csv'), index=False)
    
    # 2. Create a tidier format specifically for Bayesian analysis
    # Convert from wide to long format for easier modeling
    id_vars = ['file_name', 'word_count']
    if 'Native_Language' in results_df.columns:
        id_vars.append('Native_Language')
    
    # Add any other metadata columns that should be preserved
    for col in results_df.columns:
        if not col.endswith('_density') and not col in ['transitions', 'frame_markers', 
                                                       'endophoric_markers', 'evidentials', 
                                                       'code_glosses', 'hedges', 'boosters', 
                                                       'attitude_markers', 'engagement_markers', 
                                                       'self_mentions', 'interactive_total', 
                                                       'interactional_total']:
            if col not in id_vars:
                id_vars.append(col)
    
    # Get all density columns for the long format conversion
    density_cols = [col for col in results_df.columns if col.endswith('_density')]
    
    # Convert to long format
    long_df = pd.melt(
        results_df,
        id_vars=id_vars,
        value_vars=density_cols,
        var_name='marker_type',
        value_name='density'
    )
    
    # Clean up marker names for easier interpretation
    long_df['marker_type'] = long_df['marker_type'].str.replace('_density', '')
    
    # Create marker category column
    def get_marker_category(marker):
        interactive = ['transitions', 'frame_markers', 'endophoric_markers', 'evidentials', 'code_glosses']
        interactional = ['hedges', 'boosters', 'attitude_markers', 'engagement_markers', 'self_mentions']
        
        if marker in interactive:
            return 'interactive'
        elif marker in interactional:
            return 'interactional'
        elif 'self_mentions_' in marker:
            return 'self_mention_subcategory'
        else:
            return 'other'
    
    long_df['marker_category'] = long_df['marker_type'].apply(get_marker_category)
    
    # Add standardized density values (z-scores) for easier prior specification
    long_df['density_z'] = (long_df['density'] - long_df['density'].mean()) / long_df['density'].std()
    
    # Save the long format data
    long_df.to_csv(os.path.join(output_dir, 'metadiscourse_long_format.csv'), index=False)
    
    # 3. Save a summary for each marker type by language group (if applicable)
    if 'Native_Language' in results_df.columns:
        summary_df = long_df.groupby(['Native_Language', 'marker_type']).agg({
            'density': ['mean', 'std', 'count', 'min', 'max'],
            'density_z': ['mean', 'std']
        }).reset_index()
        
        summary_df.columns = ['_'.join(col).strip('_') for col in summary_df.columns.values]
        summary_df.to_csv(os.path.join(output_dir, 'metadiscourse_summary_by_language.csv'), index=False)
    
    # 4. Save marker correlations for defining priors
    # This helps with specifying multivariate priors in Bayesian models
    marker_corr = results_df[density_cols].corr()
    marker_corr.to_csv(os.path.join(output_dir, 'marker_correlations.csv'))
    
    print(f"Parsed metadiscourse data saved to {output_dir}/")
    return long_df

In [ ]:
def save_parsed_output(results_df, output_dir='intermediate_data'):
    """Save parsed metadiscourse data in a format optimized for Bayesian analysis.
    
    This function creates a structured dataset with all necessary information
    for running Bayesian regression models on metadiscourse markers.
    
    Args:
        results_df (pd.DataFrame): DataFrame containing metadiscourse analysis results
        output_dir (str): Directory to save the processed data
        
    Returns:
        pd.DataFrame: The long-format data optimized for Bayesian analysis
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Save the complete processed dataframe
    results_df.to_csv(os.path.join(output_dir, 'complete_metadiscourse_data.csv'), index=False)
    
    # 2. Create a tidier format specifically for Bayesian analysis
    # Convert from wide to long format for easier modeling
    id_vars = ['file_name', 'word_count']
    if 'Native_Language' in results_df.columns:
        id_vars.append('Native_Language')
    
    # Add any other metadata columns that should be preserved
    for col in results_df.columns:
        if not col.endswith('_density') and not col in ['transitions', 'frame_markers', 
                                                       'endophoric_markers', 'evidentials', 
                                                       'code_glosses', 'hedges', 'boosters', 
                                                       'attitude_markers', 'engagement_markers', 
                                                       'self_mentions', 'interactive_total', 
                                                       'interactional_total']:
            if col not in id_vars and col in results_df.columns:
                id_vars.append(col)
    
    # Get all density columns for the long format conversion
    density_cols = [col for col in results_df.columns if col.endswith('_density')]
    
    # Convert to long format
    long_df = pd.melt(
        results_df,
        id_vars=id_vars,
        value_vars=density_cols,
        var_name='marker_type',
        value_name='density'
    )
    
    # Clean up marker names for easier interpretation
    long_df['marker_type'] = long_df['marker_type'].str.replace('_density', '')
    
    # Create marker category column
    def get_marker_category(marker):
        interactive = ['transitions', 'frame_markers', 'endophoric_markers', 'evidentials', 'code_glosses']
        interactional = ['hedges', 'boosters', 'attitude_markers', 'engagement_markers', 'self_mentions']
        
        if marker in interactive:
            return 'interactive'
        elif marker in interactional:
            return 'interactional'
        elif 'self_mentions_' in marker:
            return 'self_mention_subcategory'
        else:
            return 'other'
    
    long_df['marker_category'] = long_df['marker_type'].apply(get_marker_category)
    
    # Add standardized density values (z-scores) for easier prior specification
    long_df['density_z'] = (long_df['density'] - long_df['density'].mean()) / long_df['density'].std()
    
    # Save the long format data
    long_df.to_csv(os.path.join(output_dir, 'metadiscourse_long_format.csv'), index=False)
    
    # 3. Save a summary for each marker type by language group (if applicable)
    if 'Native_Language' in results_df.columns:
        try:
            summary_df = long_df.groupby(['Native_Language', 'marker_type']).agg({
                'density': ['mean', 'std', 'count', 'min', 'max'],
                'density_z': ['mean', 'std']
            }).reset_index()
            
            summary_df.columns = ['_'.join(col).strip('_') for col in summary_df.columns.values]
            summary_df.to_csv(os.path.join(output_dir, 'metadiscourse_summary_by_language.csv'), index=False)
        except Exception as e:
            print(f"Warning: Could not create language group summary: {e}")
    
    # 4. Save marker correlations for defining priors
    # This helps with specifying multivariate priors in Bayesian models
    marker_corr = results_df[density_cols].corr()
    marker_corr.to_csv(os.path.join(output_dir, 'marker_correlations.csv'))
    
    print(f"Parsed metadiscourse data saved to {output_dir}/")
    return long_df

def run_analysis_notebook(csv_path, output_dir='results', intermediate_dir='intermediate_data', batch_size=100):
    """
    Jupyter/Colab-friendly version of run_analysis that doesn't require command line arguments.
    
    Args:
        csv_path (str): Path to CSV file with texts to analyze
        output_dir (str): Output directory for analysis results
        intermediate_dir (str): Directory to save intermediate data for Bayesian analysis
        batch_size (int): Batch size for processing texts
        
    Returns:
        pd.DataFrame: DataFrame containing the analysis results
    """
    # First ensure custom components are registered
    if not Language.has_factory("metadiscourse_detector"):
        Language.factory("metadiscourse_detector")(metadiscourse_detector)
    
    if not Language.has_factory("self_mention_categorizer"):
        Language.factory("self_mention_categorizer")(self_mention_categorizer)
    
    # Load and preprocess data
    try:
        meta = pd.read_csv(csv_path)
        print(f"Loaded {len(meta)} texts from CSV file.")
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return None
    
    # Assume the CSV has a column named 'text_field'
    if 'text_field' not in meta.columns:
        print("Error: CSV file must contain a 'text_field' column with the texts to analyze.")
        return None
    
    meta['text_field'] = meta['text_field'].apply(lambda x: re.sub(pattern, '', str(x)))
    meta['text_field'] = meta['text_field'].str.lower()
    
    # Create list of (text, metadata) tuples
    meta_records = meta.to_dict('records')
    texts = meta['text_field'].tolist()
    corpus_data = list(zip(texts, meta_records))
    
    results = []
    from tqdm import tqdm
    
    # Process in batches for better memory management
    total_batches = (len(corpus_data) + batch_size - 1) // batch_size
    print(f"Processing {len(corpus_data)} texts in {total_batches} batches (batch size: {batch_size})...")
    
    for i in tqdm(range(0, len(corpus_data), batch_size), desc="Processing text batches"):
        batch = corpus_data[i:i+batch_size]
        batch_results = []
        
        for text, metadata in batch:
            try:
                # Make sure the NLP pipeline is properly initialized
                if "metadiscourse_detector" not in nlp.pipe_names:
                    nlp.add_pipe("metadiscourse_detector", last=True)
                
                if "self_mention_categorizer" not in nlp.pipe_names:
                    nlp.add_pipe("self_mention_categorizer", last=True)
                
                # Analyze the text
                analysis = analyze_text(text, nlp)
                
                # Add metadata to analysis results
                for key, value in metadata.items():
                    if key != 'text_field':  # Avoid duplicating the text content
                        analysis[key] = value
                
                batch_results.append(analysis)
            except Exception as e:
                print(f"Error processing text: {e}")
        
        # Extend results with current batch
        results.extend(batch_results)
    
    results_df = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    results_csv = os.path.join(output_dir, 'metadiscourse_analysis_results.csv')
    results_df.to_csv(results_csv, index=False)
    print(f"Raw analysis results saved to {results_csv}")
    
    # Save the processed data in a format optimized for Bayesian analysis
    os.makedirs(intermediate_dir, exist_ok=True)
    print("Saving parsed data for Bayesian analysis...")
    long_df = save_parsed_output(results_df, output_dir=intermediate_dir)
    
    # Generate analyses and visualizations
    print("Generating LaTeX tables...")
    generate_latex_tables(results_df, output_dir)
    
    print("Performing language group statistics...")
    perform_language_group_statistics(results_df, output_dir)
    
    print("Generating visualizations...")
    generate_visualizations(results_df, output_dir)
    
    print("Analyzing metadiscourse distribution...")
    distribution_summary = analyze_metadiscourse_distribution(results_df, output_dir)
    
    print("Calculating entropy measures...")
    entropy_summary = calculate_shannon_entropy(results_df, output_dir)
    
    print(f"All analysis results saved to {output_dir}")
    
    # Print summary statistics
    print("\n=== ENHANCED METADISCOURSE ANALYSIS SUMMARY ===\n")
    print(f"Total texts analyzed: {len(results_df)}")
    if 'Native_Language' in results_df.columns:
        print(f"Language groups: {', '.join(results_df['Native_Language'].unique())}")
    
    print("\n--- Metadiscourse Distribution ---")
    print(f"Interactive markers: {distribution_summary['interactive_percentage']:.2f}%")
    print(f"Interactional markers: {distribution_summary['interactional_percentage']:.2f}%")
    print(f"Most frequent category: {distribution_summary['most_frequent_category']} ({distribution_summary['most_frequent_percentage']:.2f}%)")
    print(f"Least frequent category: {distribution_summary['least_frequent_category']} ({distribution_summary['least_frequent_percentage']:.2f}%)")
    
    # Print self-mention sub-category statistics if available
    if 'self_mentions_organizational_pct' in distribution_summary:
        print("\n--- Self-Mention Functions Distribution ---")
        print(f"Organizational: {distribution_summary['self_mentions_organizational_pct']:.2f}%")
        print(f"Stance-taking: {distribution_summary['self_mentions_stance_pct']:.2f}%")
        print(f"Methodological: {distribution_summary['self_mentions_methodological_pct']:.2f}%")
        print(f"Other uses: {distribution_summary['self_mentions_other_pct']:.2f}%")
    
    print("\n--- Metadiscourse Diversity (Shannon Entropy) ---")
    print(f"Mean normalized entropy: {entropy_summary['mean_normalized_entropy']:.4f} (0-1 scale)")
    print(f"Max possible entropy: {entropy_summary['max_entropy']:.4f} bits")
    
    if 'self_mentions_normalized_entropy' in entropy_summary:
        print(f"Self-mention functional diversity: {entropy_summary['self_mentions_normalized_entropy']:.4f} (0-1 scale)")
    
    return results_df, long_df



# Example usage in a Jupyter notebook:
results_df, long_df = run_analysis_notebook(
     csv_path="/Users/fatihbozdag/Documents/Studies/AI Library/Metadata/metadata_with_text.csv",
    output_dir="/Users/fatihbozdag/Documents/Studies/MetadiscourseMarkers/results/analysis_results/",
     intermediate_dir="intermediate_data",
     batch_size=500
 )

In [ ]:
# Copy this improved visualization function to a new cell in your notebook

def generate_visualizations_fixed(df, output_dir='results'):
    """Generate improved visualizations with explicit file saving."""
    import os
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np
    from pathlib import Path
    
    # Ensure output directory exists
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Saving visualizations to {output_dir.absolute()}")
    
    # Set up plotting environment
    plt.rcParams.update({
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'figure.figsize': (12, 8),
        'figure.dpi': 300
    })
    
    sns.set(style="whitegrid")
    
    # Track saved files to report back
    saved_files = []
    
    # 1. Scatter Plot: Interactive vs. Interactional Markers
    print("Generating scatter plot...")
    plt.figure(figsize=(10, 8))
    
    try:
        if 'Native_Language' in df.columns:
            scatter = sns.scatterplot(
                data=df,
                x='interactive_density',
                y='interactional_density',
                hue='Native_Language',
                s=100,
                alpha=0.7
            )
        else:
            scatter = sns.scatterplot(
                data=df,
                x='interactive_density',
                y='interactional_density',
                s=100,
                alpha=0.7
            )
            
        plt.title('Interactive vs. Interactional Metadiscourse Markers')
        plt.xlabel('Interactive Markers (per 1000 words)')
        plt.ylabel('Interactional Markers (per 1000 words)')
        plt.grid(True, linestyle='--', alpha=0.7)
        
        # Add regression line
        sns.regplot(
            x='interactive_density',
            y='interactional_density',
            data=df,
            scatter=False,
            ci=None,
            line_kws={"color": "red", "lw": 2, "linestyle": "--"}
        )
        
        plt.tight_layout()
        
        # Save with explicit filepath
        scatter_path = output_dir / 'interactive_vs_interactional.png'
        plt.savefig(scatter_path, bbox_inches='tight')
        saved_files.append(str(scatter_path))
        print(f"Saved scatter plot to {scatter_path}")
        
        plt.close()
    except Exception as e:
        print(f"Error generating scatter plot: {e}")
    
    # 2. Boxplot of Main Marker Categories
    print("Generating boxplot...")
    try:
        main_density_cols = [col for col in df.columns if col.endswith('_density') and 
                           not col.startswith('interactive_') and 
                           not col.startswith('interactional_') and
                           not 'self_mentions_' in col]
        
        if main_density_cols:
            plt.figure(figsize=(14, 8))
            
            id_vars = []
            if 'Native_Language' in df.columns:
                id_vars.append('Native_Language')
                
            melted_df = pd.melt(
                df,
                id_vars=id_vars,
                value_vars=main_density_cols,
                var_name='Marker_Category',
                value_name='Density'
            )
            
            melted_df['Marker_Category'] = melted_df['Marker_Category'].str.replace('_density', '')
            
            # Create boxplot and save
            ax = sns.boxplot(
                x='Marker_Category',
                y='Density',
                data=melted_df,
                palette="Set3"
            )
            
            plt.xticks(rotation=45, ha='right')
            plt.title('Distribution of Metadiscourse Marker Categories')
            plt.xlabel('Marker Category')
            plt.ylabel('Density (per 1000 words)')
            plt.tight_layout()
            
            box_path = output_dir / 'marker_categories_boxplot.png'
            plt.savefig(box_path, bbox_inches='tight')
            saved_files.append(str(box_path))
            print(f"Saved boxplot to {box_path}")
            
            plt.close()
    except Exception as e:
        print(f"Error generating boxplot: {e}")
    
    # 3. Word Count Histogram
    print("Generating word count histogram...")
    try:
        plt.figure(figsize=(10, 6))
        ax = sns.histplot(df['word_count'], bins=30, kde=True, color='skyblue')
        plt.title("Distribution of Word Counts")
        plt.xlabel("Word Count")
        plt.ylabel("Frequency")
        plt.tight_layout()
        
        hist_path = output_dir / 'word_count_histogram.png'
        plt.savefig(hist_path, bbox_inches='tight')
        saved_files.append(str(hist_path))
        print(f"Saved histogram to {hist_path}")
        
        plt.close()
    except Exception as e:
        print(f"Error generating word count histogram: {e}")
    
    # 4. Heatmap of correlations
    print("Generating correlation heatmap...")
    try:
        corr_cols = [col for col in df.columns if col.endswith('_density')]
        if len(corr_cols) > 1:  # Need at least 2 columns for correlation
            corr = df[corr_cols].corr()
            
            # Clean up column names for display
            corr.columns = [col.replace('_density', '') for col in corr.columns]
            corr.index = [idx.replace('_density', '') for idx in corr.index]
            
            plt.figure(figsize=(14, 12))
            mask = np.triu(np.ones_like(corr, dtype=bool))  # Create mask for upper triangle
            
            # Generate heatmap
            heatmap = sns.heatmap(
                corr, 
                annot=True,
                fmt=".2f",
                cmap="coolwarm",
                mask=mask,
                vmin=-1, 
                vmax=1,
                square=True,
                linewidths=0.5
            )
            
            plt.title("Correlation Between Metadiscourse Markers", fontsize=18)
            plt.tight_layout()
            
            heatmap_path = output_dir / 'marker_correlation_heatmap.png'
            plt.savefig(heatmap_path, bbox_inches='tight')
            saved_files.append(str(heatmap_path))
            print(f"Saved correlation heatmap to {heatmap_path}")
            
            plt.close()
    except Exception as e:
        print(f"Error generating correlation heatmap: {e}")
    
    # Report results
    if saved_files:
        print(f"\nSuccessfully saved {len(saved_files)} visualization files:")
        for path in saved_files:
            print(f"- {path}")
    else:
        print("Warning: No visualization files were saved!")
        
    return saved_files

# Load your existing analysis results
results_path = "/Users/fatihbozdag/Documents/Studies/results/metadiscourse_analysis_results.csv"
output_dir = "/Users/fatihbozdag/Documents/Studies/MetadiscourseMarkers/results"

import pandas as pd
import os

# Check if the results file exists
if os.path.exists(results_path):
    print(f"Loading analysis results from {results_path}")
    df = pd.read_csv(results_path)
    print(f"Loaded data with {len(df)} rows and {len(df.columns)} columns")
    
    # Generate and save visualizations
    saved_files = generate_visualizations_fixed(df, output_dir)
    
    # Print summary of saved visualizations
    if saved_files:
        print("\nVisualization files have been saved to:")
        print(output_dir)
    else:
        print("\nNo visualization files were saved.")
        
    # Print directory contents to verify files
    print("\nContents of the output directory:")
    try:
        files = os.listdir(output_dir)
        for file in files:
            if file.endswith(".png"):
                file_path = os.path.join(output_dir, file)
                file_size = os.path.getsize(file_path) / 1024  # Size in KB
                print(f"- {file} ({file_size:.1f} KB)")
    except Exception as e:
        print(f"Error listing directory contents: {e}")
else:
    print(f"Error: Results file not found at {results_path}")
    print("Please check the file path and try again.")